# CT-ScanEval: run the models in Google Colab (free)

1. Get a free Gemini API key at https://aistudio.google.com/apikey (Google account, no card).
2. Add it as a Colab **Secret** (key icon in the left sidebar) named `GEMINI_API_KEY`, with notebook access on. Paid keys (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`) are optional.
3. Add `GITHUB_TOKEN` too (see the last section) so results are saved back to GitHub.
4. Runtime, Run all. The CPU runtime is enough. Nothing needs to be uploaded.

In [ ]:
!git clone https://github.com/Moyo8/-CT-ScanEval /content/CT-ScanEval
%cd /content/CT-ScanEval

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q tests   # grader checks: should say 60 passed

In [ ]:
import os
from google.colab import userdata
for k in ['GEMINI_API_KEY', 'ANTHROPIC_API_KEY', 'OPENAI_API_KEY']:
    try:
        os.environ[k] = userdata.get(k)
        print(k, 'loaded')
    except Exception:
        print(k, 'not set (skip that provider)')

## Run the models
Each run is 8 tasks x 3 trials = 24 calls, made one at a time 7 s apart to stay under free-tier per-minute limits (about 5 min per model). If the free daily quota runs out, the run stops cleanly and keeps the finished trials. Your current limits are at https://aistudio.google.com/rate-limit

In [ ]:
# Gemini 3.8 Flash: Google's newest Flash model (free tier)
!python run_eval.py --provider gemini --model gemini-3.8-flash --trials 3 --pace 7

In [ ]:
# Gemini 3.5 Flash-Lite: smaller, cheaper model for comparison (free tier)
!python run_eval.py --provider gemini --model gemini-3.5-flash-lite --trials 3 --pace 7

In [ ]:
# Optional, PAID: Claude or OpenAI if you have a key
# !python run_eval.py --provider anthropic --model claude-opus-5-5 --effort high --trials 3
# !python run_eval.py --provider openai --model YOUR_OPENAI_MODEL_ID --trials 3

## Baselines and summary

In [ ]:
!python run_eval.py --provider random --trials 3
!python analysis/summarize.py --latest
from IPython.display import Image, display
display(Image('results/charts/per_task_scores.png'))

## Save results to GitHub
Colab deletes its files when the session ends, so this pushes `results/` and `transcripts/` back to the repository.

One-time setup: create a GitHub token (GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → Generate). Give it access to **only** the `-CT-ScanEval` repository with **Contents: Read and write**, then add it as a Colab Secret named `GITHUB_TOKEN`.

In [ ]:
import base64, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
def git(*args):
    r = subprocess.run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').replace(auth, '***').strip())
    return r.returncode
git('config', 'user.name', 'Moyo8')
git('config', 'user.email', 'Moyo8@users.noreply.github.com')
git('add', 'results', 'transcripts')
git('commit', '-m', 'Model runs from Colab')
git('pull', '--rebase', 'origin', 'main')
print('pushed OK' if git('push', 'origin', 'HEAD:main') == 0 else 'PUSH FAILED - check the token')